In [18]:
import pandas as pd

## Logements sociaux

KPI pour ce dataset

- Surface
- Type de logement
- Logements financés
- A qui c'est destiné : exemple : familles
- Ancienneté de l'immeuble

Si besoin on peut trouver des datasets pour les aides au logement

**Que 2 colonnes à encoder :**
- Mode de réalisation
- Nature de programme

**Signification de Mode de réalisation :**

- la construction neuve
- l’acquisition suivie d’une réhabilitation lourde et de la restructuration d’immeubles assimilable à de la construction neuve
- l’acquisition et le conventionnement en logement social avec pas ou peu de travaux.

https://opendata.paris.fr/explore/dataset/logements-sociaux-finances-a-paris/information/?disjunctive.bs&disjunctive.mode_real&disjunctive.nature_programme&disjunctive.ville&disjunctive.code_postal

In [19]:
df_hlm = pd.read_csv('../../data/bronze/immobilier/hlm.csv', sep = ";")

In [20]:
df_hlm['Nature de programme'].value_counts()

# Mode de réalisation
# neuve = 0
# acquisition réhabilitation = 1
# aquisition conventionnement = 2

# Nature de programme
#


Nature de programme
logement familial                                                                    3551
résidence étudiante                                                                   173
foyer pour jeunes travailleurs ou résidence pour jeunes actifs                        114
maison relais                                                                          84
résidence sociale pour démunis                                                         81
foyer de travailleurs migrants                                                         36
résidence sociale issue du plan de traitement des foyers de travailleurs migrants      33
foyer pour personnes handicapées                                                       32
établissement d'hébergement pour personnes âgées dépendantes                           26
centre d'hébergement d'urgence                                                         26
résidence pour personnes âgées                                                  

In [21]:
# on applatit la colonne geo_point_2d
df_hlm[["Longitude", "Latitude"]] = df_hlm["geo_point_2d"].str.split(",", expand=True)

In [22]:
# on drop les colonnes inutiles
df_hlm = df_hlm.drop(columns = ["Bailleur social","Commentaires", "Ville", "geo_shape", "geo_point_2d"])

In [23]:
# pas de na
df_hlm.head()

,Identifiant livraison,Adresse du programme,Code postal,Année du financement - agrément,Nombre total de logements financés,Dont nombre de logements PLA I,Dont nombre de logements PLUS,Dont nombre de logements PLUS CD,Dont nombre de logements PLS,Mode de réalisation,Arrondissement,Nature de programme,Coordonnée en X (L93),Coordonnée en Y (L93),Longitude,Latitude
0,2000011,16-20 RUE DES MEUNIERS,75012,2001,62,0,62,0,0,acquisition conventionnement,12,logement familial,655823.1529,6.859495e+06,48.83399817959342,2.398175442412243
1,2000035,25 RUE DES ANNELETS,75019,2001,18,10,8,0,0,acquisition réhabilitation,19,logement familial,655225.8330,6.864433e+06,48.8783619998301,2.3895183752327043
2,2000044,LOT D1B - 86 RUE DES HAIES,75020,2001,6,0,6,0,0,acquisition réhabilitation,20,logement familial,656258.4385,6.861828e+06,48.85500366971155,2.403865260493333
3,2000048,31 QUAI DE VALMY,75010,2001,28,0,22,0,6,acquisition conventionnement,10,logement familial,653551.4260,6.863384e+06,48.86880747974437,2.366804047083707
4,2000062,19 RUE DES PLANTES,75014,2001,38,0,15,0,23,acquisition conventionnement,14,logement familial,650356.4410,6.859151e+06,48.83049983027532,2.3237456731730535


### Loyers

KPI pour ce dataset

- Nombre de pièces
- Si c'est meublé ou non
- Ancienneté de l'immeuble
- Loyers de référence

**Voir comment encoder Epoque de construction** 

In [24]:
df_loyers = pd.read_csv('../../data/bronze/immobilier/loyers.csv', sep = ";")

In [25]:
# on applatit la colonne geo_point_2d
df_loyers[["Longitude", "Latitude"]] = df_loyers["geo_point_2d"].str.split(",", expand=True)

In [26]:
# on drop la colonne ville elle n'a qu'une seul valeur : Paris
df_loyers = df_loyers.drop(columns = ["Ville", "geo_shape", "geo_point_2d"])

In [27]:
df_loyers['Epoque de construction'].value_counts()

# Type de location
# non meublé : 0
# meublé : 1

# Epoque de construction

Epoque de construction
1946-1970     4480
Avant 1946    4480
1971-1990     4480
Apres 1990    4480
Name: count, dtype: int64

In [28]:
# encodage de la colonne binaire
# p-e il faut changer le type en bool
df_loyers['Type de location'] = df_loyers['Type de location'].map({"meublé": 1, "non meublé": 0})

In [29]:
# on renomme la colonne pour un nom plus explicit
df_loyers = df_loyers.rename(columns={'Type de location': "Meublé"})

In [30]:
df_loyers.head()

,Année,Secteurs géographiques,Numéro du quartier,Nom du quartier,Nombre de pièces principales,Epoque de construction,Meublé,Loyers de référence,Loyers de référence majorés,Loyers de référence minorés,Numéro INSEE du quartier,Longitude,Latitude
0,2023,11,77,Belleville,3,1946-1970,0,19.8,23.80,13.90,7512077,48.87153120058614,2.3875492398498035
1,2023,2,21,Monnaie,2,Avant 1946,1,33.0,39.60,23.10,7510621,48.85438440363996,2.340035371130514
2,2023,4,12,Sainte-Avoie,1,1971-1990,1,34.8,41.80,24.40,7510312,48.862557245043206,2.354851518245669
3,2020,4,19,Val-de-Grace,2,Avant 1946,1,31.3,37.56,21.91,7510519,48.84168428795317,2.3438609263160095
4,2020,10,18,Jardin-des-Plantes,4,Avant 1946,1,23.3,27.96,16.31,7510518,48.84194019337512,2.356893889624744


## KPI metier calcules automatiquement

Cette section calcule des KPI metier concrets sur les datasets:
- Loyers (loyer median/moyen, directement par quartier)
- Logements sociaux HLM (repartition par arrondissement)


In [31]:
from pathlib import Path
import re
import unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

DATA_DIR = Path("../../data/bronze/immobilier/")

df_hlm_kpi = pd.read_csv(DATA_DIR / "hlm.csv", sep=";")
df_loyers_kpi = pd.read_csv(DATA_DIR / "loyers.csv", sep=";")


def normalize_text(text: str) -> str:
    text = str(text).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


def pick_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    normalized_map = {normalize_text(col): col for col in df.columns}

    for candidate in candidates:
        key = normalize_text(candidate)
        if key in normalized_map:
            return normalized_map[key]

    for candidate in candidates:
        key = normalize_text(candidate)
        for normalized_col, original_col in normalized_map.items():
            if key in normalized_col:
                return original_col

    return None


def to_numeric(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype(str)
        .str.replace("\xa0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )
    return pd.to_numeric(cleaned, errors="coerce")


def arr_from_postal_code(series: pd.Series) -> pd.Series:
    code = to_numeric(series).astype("Int64")
    arr = code - 75000
    arr = arr.where((arr >= 1) & (arr <= 20))
    return arr


def fmt(value: float | int | None, digits: int = 2):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    if digits == 0:
        return int(round(float(value), 0))
    return round(float(value), digits)

In [32]:
# 1) KPI loyers par quartier (calcul direct sur les données source)
EXPORT_DIR = Path("../../data/outputs")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
col_annee = pick_column(df_loyers_kpi, ["Année", "annee"])
col_loyer_ref = pick_column(df_loyers_kpi, ["loyers de reference", "loyers_de_reference"])
col_loyer_maj = pick_column(df_loyers_kpi, ["loyers de reference majores", "loyers_de_reference_majores"])
col_loyer_min = pick_column(df_loyers_kpi, ["loyers de reference minores", "loyers_de_reference_minores"])
col_arr_loyer = pick_column(df_loyers_kpi, ["secteurs geographiques", "arrondissement"])
col_num_quartier = pick_column(df_loyers_kpi, ["numero du quartier", "numéro du quartier"])
col_nom_quartier = pick_column(df_loyers_kpi, ["nom du quartier"])
col_insee_quartier = pick_column(df_loyers_kpi, ["numero insee du quartier", "numéro insee du quartier"])

for col in [col_loyer_ref, col_loyer_maj, col_loyer_min, col_arr_loyer]:
    if col:
        df_loyers_kpi[col] = to_numeric(df_loyers_kpi[col])

group_cols_loyer = [c for c in [col_annee, col_arr_loyer, col_num_quartier, col_nom_quartier, col_insee_quartier] if c]

if group_cols_loyer and col_loyer_ref:
    agg_map = {
        "loyer_reference_median": (col_loyer_ref, "median"),
        "loyer_reference_moyen": (col_loyer_ref, "mean"),
        "nb_observations": (col_loyer_ref, "size"),
    }
    if col_loyer_maj:
        agg_map["loyer_majore_median"] = (col_loyer_maj, "median")
        agg_map["loyer_majore_moyen"] = (col_loyer_maj, "mean")
    if col_loyer_min:
        agg_map["loyer_minore_median"] = (col_loyer_min, "median")
        agg_map["loyer_minore_moyen"] = (col_loyer_min, "mean")

    rename_map = {}
    if col_annee: rename_map[col_annee] = "annee"
    if col_arr_loyer: rename_map[col_arr_loyer] = "arrondissement"
    if col_num_quartier: rename_map[col_num_quartier] = "numero_quartier"
    if col_nom_quartier: rename_map[col_nom_quartier] = "nom_quartier"
    if col_insee_quartier: rename_map[col_insee_quartier] = "code_insee_quartier"

    kpi_loyers_quartier = (
        df_loyers_kpi.dropna(subset=[group_cols_loyer[0], col_loyer_ref])
        .groupby(group_cols_loyer, as_index=False)
        .agg(**agg_map)
        .rename(columns=rename_map)
        .sort_values(["arrondissement", "numero_quartier"] if "numero_quartier" in [rename_map.get(c) for c in group_cols_loyer] else ["arrondissement"])
    )

    kpi_loyers_quartier["arrondissement"] = kpi_loyers_quartier["arrondissement"].astype("Int64")
    for c in ["loyer_reference_median", "loyer_reference_moyen",
              "loyer_majore_median", "loyer_majore_moyen",
              "loyer_minore_median", "loyer_minore_moyen"]:
        if c in kpi_loyers_quartier.columns:
            kpi_loyers_quartier[c] = kpi_loyers_quartier[c].round(2)

    display(kpi_loyers_quartier)
    kpi_loyers_quartier.to_csv(EXPORT_DIR / "kpi_loyers_quartier.csv", index=False)
    print(f"kpi_loyers_quartier.csv : {len(kpi_loyers_quartier)} lignes")
else:
    print("KPI loyers indisponible (colonnes manquantes).")

# 2) KPI logements sociaux HLM — pas de colonne quartier, reste par arrondissement
col_arr_hlm = pick_column(df_hlm_kpi, ["arrondissement"])
col_log_fin = pick_column(df_hlm_kpi, ["nombre total de logements finances", "nombre_total_de_logements_finances"])

if col_log_fin:
    df_hlm_kpi[col_log_fin] = to_numeric(df_hlm_kpi[col_log_fin])

if col_arr_hlm and col_log_fin:
    hlm_par_arr = (
        df_hlm_kpi.dropna(subset=[col_arr_hlm, col_log_fin])
        .groupby(col_arr_hlm, as_index=False)
        .agg(
            logements_finances_total=(col_log_fin, "sum"),
            logements_finances_moyen=(col_log_fin, "mean"),
            nb_programmes=(col_log_fin, "size"),
        )
        .rename(columns={col_arr_hlm: "arrondissement"})
        .sort_values("arrondissement")
    )
    hlm_par_arr["logements_finances_moyen"] = hlm_par_arr["logements_finances_moyen"].round(2)
    display(hlm_par_arr)
    hlm_par_arr.to_csv(EXPORT_DIR / "kpi_repartition_logements_sociaux.csv", index=False)
    print(f"kpi_repartition_logements_sociaux.csv : {len(hlm_par_arr)} lignes")
else:
    print("KPI logements sociaux indisponible (colonne arrondissement ou logements finances manquante).")

print(f"\nCSV KPI generes dans: {EXPORT_DIR.resolve()}")


,annee,arrondissement,numero_quartier,nom_quartier,code_insee_quartier,loyer_reference_median,loyer_reference_moyen,nb_observations,loyer_majore_median,loyer_majore_moyen,loyer_minore_median,loyer_minore_moyen
0,2019,1,23,Notre-Dame-des-Champs,7510623,29.40,29.58,32,35.30,35.50,20.60,20.72
80,2020,1,23,Notre-Dame-des-Champs,7510623,29.85,30.19,32,35.82,36.23,20.90,21.13
160,2021,1,23,Notre-Dame-des-Champs,7510623,30.40,30.77,32,36.48,36.92,21.28,21.54
240,2022,1,23,Notre-Dame-des-Champs,7510623,31.00,31.21,32,37.20,37.45,21.70,21.85
320,2023,1,23,Notre-Dame-des-Champs,7510623,31.45,31.43,32,37.75,37.72,22.05,22.01
...,...,...,...,...,...,...,...,...,...,...,...,...
239,2021,14,79,Père-Lachaise,7512079,22.75,22.79,32,27.30,27.35,15.92,15.95
319,2022,14,79,Père-Lachaise,7512079,23.10,22.92,32,27.72,27.51,16.17,16.05
399,2023,14,79,Père-Lachaise,7512079,23.40,23.06,32,28.10,27.68,16.40,16.15
479,2024,14,79,Père-Lachaise,7512079,23.80,23.48,32,28.60,28.18,16.70,16.44


kpi_loyers_quartier.csv : 560 lignes


,arrondissement,logements_finances_total,logements_finances_moyen,nb_programmes
0,1,877,12.18,72
1,2,658,11.54,57
2,3,1336,14.84,90
3,4,1677,17.11,98
4,5,2194,25.51,86
5,6,920,17.69,52
6,7,805,27.76,29
7,8,926,20.13,46
8,9,2097,17.33,121
9,10,4859,21.60,225


kpi_repartition_logements_sociaux.csv : 20 lignes

CSV KPI generes dans: C:\Users\rayan\Desktop\efrei\projet-data-dev\data\outputs
